# 第 28 课 · 真实开源小模型、配套分词器与 Chat Template

**从自己写的微型模型，走向阅读和调用真实模型。**

本课选用 `Qwen/Qwen3-0.6B`，不是声称它最新或最强，而是把模型、配套 tokenizer 和生成接口接起来。保留前几课的形状、GQA、RoPE、RMSNorm、KV cache 知识，验证实际模型有哪些相同点和不同点。[S1][S2]

**产出：** 一次可记录的本地推理；能解释消息怎样变成 ID，怎样从 `generate()` 的返回值中只取回答。

> **本讲义的验证边界**：制作环境有 PyTorch，但没有 Transformers，且执行环境不能下载依赖/权重。因此随文件保留的运行输出只有离线实验。真实 tokenizer / Qwen3 权重 / CUDA 推理单元默认关闭，需在你的电脑打开。没有伪造真实模型回答、速度或显存数据。离线替身绝不是小型 Qwen。

本 Notebook 已内置所需函数，不依赖配套 `.py` 或之前课程。正文按顺序学习，末尾包含独立练习和参考答案。

## 0. 先决定要运行哪一层

| 层次 | 目的 | 需要什么 |
|---|---|---|
| 离线讲解与检查（默认） | 形状、消息格式、左补齐、输出切片、状态恢复 | 已有 PyTorch |
| 真实分词器检查 | 下载小型配置和 tokenizer，比较模板编码路线 | Transformers + 网络，或完整缓存 |
| 真实模型生成 | 下载并加载权重，在本地计算回答 | 同上 + 足够 RAM/显存 |

继续使用已有 PyTorch 环境，不要为了本课重装整个 CUDA 栈。安装新依赖时，在**这个 Notebook 使用的同一个 Python 环境**执行：

```bash
python -m pip install "transformers>=4.57,<6" safetensors
```

这是按官方接口编写的依赖范围，**不是本制作环境已安装实测的版本组合**。完整运行会记录你的实际版本。原模型卡说明 Qwen3 至少需要 Transformers 4.51，本课使用较新的 `dtype` 等接口。[S1][S6]

Notebook 中可以使用 `%pip install "transformers>=4.57,<6" safetensors`，安装后重启内核；不要同时启动脚本与 Notebook 加载两份权重。

GPU PyTorch 的安装请以 [官方安装选择器](https://pytorch.org/get-started/locally/) 为准。默认 CPU 用 FP32；CUDA 根据支持情况选 BF16/FP16。第一次真实运行会下载 **GB 级权重**；下载量、参数载荷和运行峰值不是同一个数字。先用单条短提示和 64 个新增 token。

In [1]:
import argparse
import copy
import hashlib
import importlib.metadata
import json
import math
import platform
import re
import sys
import tempfile
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace
from typing import Any

MODEL_ID = "Qwen/Qwen3-0.6B"
REVISION = "c1899de289a04d12100db370d81485cdf75e47ca"
FORMAT = "lesson28-real-model-v1"
SOURCES = {
    "model_card": "https://huggingface.co/Qwen/Qwen3-0.6B",
    "model_revision": "https://huggingface.co/api/models/Qwen/Qwen3-0.6B",
    "config": f"https://huggingface.co/Qwen/Qwen3-0.6B/blob/{REVISION}/config.json",
    "chat": "https://huggingface.co/docs/transformers/chat_templating",
    "generation": "https://huggingface.co/docs/transformers/v4.57.3/en/llm_tutorial",
    "generation_config": "https://huggingface.co/docs/transformers/v4.57.3/en/main_classes/text_generation",
    "loading": "https://huggingface.co/docs/transformers/v4.57.3/en/main_classes/model",
    "qwen_source": "https://github.com/huggingface/transformers/blob/v4.57.3/src/transformers/models/qwen3/modeling_qwen3.py",
    "install": "https://pytorch.org/get-started/locally/",
}
# 来自官方 config 的小型事实快照；不是本机加载模型后的输出。
CONFIG_SNAPSHOT = {
    "model_type": "qwen3", "hidden_size": 1024, "num_hidden_layers": 28,
    "num_attention_heads": 16, "num_key_value_heads": 8, "head_dim": 128,
    "intermediate_size": 3072, "vocab_size": 151936,
    "max_position_embeddings": 40960, "rope_theta": 1000000,
    "tie_word_embeddings": True,
}


def environment_info() -> dict:
    packages = {}
    for name in ("torch", "transformers", "tokenizers", "huggingface_hub", "safetensors"):
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = None
    return {"python": platform.python_version(), "platform": platform.platform(),
            "packages": packages}


def require_torch():
    try:
        import torch
        return torch
    except ImportError as exc:
        raise RuntimeError("本课需已有 PyTorch。请沿用前几课环境；GPU 安装参考官方安装选择器。") from exc


def require_transformers():
    try:
        import transformers
        from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, GenerationConfig
    except ImportError as exc:
        raise RuntimeError(
            "真实模型路径需要 transformers。请在当前内核的 Python 环境安装："
            "python -m pip install 'transformers>=4.57,<6' safetensors；安装后重启内核。"
        ) from exc
    version = tuple(int(x) for x in re.findall(r"\d+", transformers.__version__)[:2])
    if version < (4, 57):
        raise RuntimeError("本课的接口基线为 transformers>=4.57；请升级或使用独立环境。")
    return AutoConfig, AutoTokenizer, AutoModelForCausalLM, GenerationConfig


def json_write(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def fingerprint(value: Any) -> str:
    encoded = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(encoded).hexdigest()

In [2]:
print(json.dumps(environment_info(), ensure_ascii=False, indent=2))
import sys
print("当前内核的 Python：", sys.executable)

{
  "python": "3.13.5",
  "platform": "Linux-6.18.57-x86_64-with-glibc2.41",
  "packages": {
    "torch": "2.10.0+cpu",
    "transformers": null,
    "tokenizers": null,
    "huggingface_hub": "1.16.1",
    "safetensors": "0.7.0"
  }
}
当前内核的 Python： /opt/pyvenv/bin/python


## 1. 模型不只是一个权重文件

真实仓库中的各部分需要一起理解：

| 文件 / 对象 | 作用 |
|---|---|
| `config.json` / `AutoConfig` | 描述层数、投影宽度、词表行数等结构 |
| `*.safetensors` / `AutoModelForCausalLM.from_pretrained` | 提供已经训练好的参数 |
| `tokenizer.json`、相关配置 / `AutoTokenizer` | 字符串与 ID 的转换规则 |
| tokenizer 中的 `chat_template` | 把角色消息排列成模型训练时认识的形式 |
| `generation_config.json` | 生成时的默认设置；不等于训练配置 |

`from_config()` 只按结构初始化一个新模型，不会自动获得训练后的能力；本课加载权重使用 `from_pretrained()`。[S6]

**使用同一仓库、同一 revision 的 config、tokenizer 和 weights。** 默认固定官方仓库的 commit，避免日后 `main` 改动而悄悄更换组件。版本和本机环境仍要记录，固定模型 commit 不能保证跨设备逐位重现。

本课不使用 `trust_remote_code=True`，只读取 safetensors 权重；这减少了执行仓库自定义 Python 的范围，但不意味着任何来源文件都绝对安全。请保留来源信任与文件校验意识。

In [3]:
print("模型：", MODEL_ID)
print("默认固定 revision：", REVISION)
print("配置快照来自官方文件，不是本机加载结果：")
print(json.dumps(CONFIG_SNAPSHOT, ensure_ascii=False, indent=2))

模型： Qwen/Qwen3-0.6B
默认固定 revision： c1899de289a04d12100db370d81485cdf75e47ca
配置快照来自官方文件，不是本机加载结果：
{
  "model_type": "qwen3",
  "hidden_size": 1024,
  "num_hidden_layers": 28,
  "num_attention_heads": 16,
  "num_key_value_heads": 8,
  "head_dim": 128,
  "intermediate_size": 3072,
  "vocab_size": 151936,
  "max_position_embeddings": 40960,
  "rope_theta": 1000000,
  "tie_word_embeddings": true
}


## 2. 阅读真实配置：不要把教学约定当成所有模型的规律

Qwen3-0.6B 配置写明：隐藏维度 1024、Query 头数 16、KV 头数 8、**head_dim=128**。[S2]

注意：`1024 / 16 = 64`，**不是 128**。前几课让所有头拼接后的宽度等于隐藏维度，是教学结构的一项选择，不是普遍定律。

据配置推导，PyTorch Linear 的权重形状为 `[out_features, in_features]`：

- Q 投影：`[2048,1024]`；K/V 投影：各 `[1024,1024]`。
- Q 合并后的 2048 维，经过 O 投影 `[1024,2048]` 回到隐藏维度 1024。

真实实现还在 Q/K 上使用 norm，并可共享 Embedding 与词表输出权重。后续加载时，我们实际读取这些属性，不仅相信手算。[S7]

**先看 config，再看源码，再看实际参数形状。**

In [4]:
def positive_int(value: int, name: str) -> int:
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} 必须是正整数，不能是 bool")
    return value


def validate_messages(messages: list[dict]) -> list[dict]:
    """本课仅支持文本：可选开头 system，随后 user/assistant 交替，最后是 user。"""
    if not isinstance(messages, list) or not messages:
        raise ValueError("messages 必须是非空列表")
    clean = []
    for message in messages:
        if not isinstance(message, dict) or set(message) != {"role", "content"}:
            raise ValueError("每条消息只包含 role 和 content；本课不支持工具/多模态消息")
        if message["role"] not in {"system", "user", "assistant"}:
            raise ValueError("不支持这个 role")
        if not isinstance(message["content"], str):
            raise TypeError("content 必须是字符串")
        message["content"].encode("utf-8", errors="strict")
        clean.append(dict(message))
    start = 1 if clean[0]["role"] == "system" else 0
    dialogue = clean[start:]
    if not dialogue or len(dialogue) % 2 != 1:
        raise ValueError("本课的生成输入必须以 user 结束")
    for i, message in enumerate(dialogue):
        if message["role"] != ("user" if i % 2 == 0 else "assistant"):
            raise ValueError("system 只能在开头；user/assistant 必须交替")
    return clean


def extend_conversation(messages: list[dict], answer: str, next_user: str) -> list[dict]:
    clean = validate_messages(messages)
    if not isinstance(answer, str) or not isinstance(next_user, str):
        raise TypeError("answer 和 next_user 都必须是字符串")
    return validate_messages(clean + [{"role": "assistant", "content": answer},
                                      {"role": "user", "content": next_user}])


def eos_ids(value) -> list[int]:
    values = [value] if type(value) is int else value
    if not isinstance(values, (list, tuple)) or not values:
        raise ValueError("必须从配套配置中取得非空 EOS ID 列表")
    if any(type(x) is not int or x < 0 for x in values):
        raise ValueError("EOS 必须是非负整数")
    return list(dict.fromkeys(values))


def check_budget(input_width: int, max_input: int, max_new: int,
                 context_limit: int) -> None:
    for name, value in (("input_width", input_width), ("max_input", max_input),
                        ("max_new", max_new), ("context_limit", context_limit)):
        positive_int(value, name)
    if input_width > max_input:
        raise ValueError(f"模板后的输入长度 {input_width} 超过本课输入上限 {max_input}；不自动截断")
    if input_width + max_new > context_limit:
        raise ValueError("模板输入长度 + 最大新生成长度超过本课上下文预算；不自动丢弃历史")


def architecture_summary(config: dict) -> dict:
    """优先读取显式 head_dim；hidden_size/Hq 仅在未提供该字段时作为回退。"""
    d = positive_int(config["hidden_size"], "hidden_size")
    hq = positive_int(config["num_attention_heads"], "Hq")
    hkv = positive_int(config.get("num_key_value_heads", hq), "Hkv")
    head = config.get("head_dim")
    if head is None:
        if d % hq:
            raise ValueError("未提供 head_dim 且 hidden_size 不能被 Hq 整除")
        head = d // hq
    positive_int(head, "head_dim")
    if hq % hkv:
        raise ValueError("本课要求 Hq 可被 Hkv 整除")
    layers = positive_int(config["num_hidden_layers"], "layers")
    return {"hidden_size": d, "layers": layers, "query_heads": hq, "kv_heads": hkv,
            "head_dim": head, "q_projection_shape": [hq * head, d],
            "k_projection_shape": [hkv * head, d],
            "v_projection_shape": [hkv * head, d],
            "o_projection_shape": [d, hq * head],
            "kv_bytes_per_token_fp16_batch1": 2 * layers * hkv * head * 2,
            "vocab_size_from_config": config["vocab_size"],
            "max_positions_config_not_quality_guarantee": config.get("max_position_embeddings")}

In [5]:
summary = architecture_summary(CONFIG_SNAPSHOT)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("朴素相除：", CONFIG_SNAPSHOT['hidden_size'] // CONFIG_SNAPSHOT['num_attention_heads'])
print("真实配置 head_dim：", summary['head_dim'])
print("FP16 KV 每token理论载荷：", summary['kv_bytes_per_token_fp16_batch1'] / 1024, "KiB")

{
  "hidden_size": 1024,
  "layers": 28,
  "query_heads": 16,
  "kv_heads": 8,
  "head_dim": 128,
  "q_projection_shape": [
    2048,
    1024
  ],
  "k_projection_shape": [
    1024,
    1024
  ],
  "v_projection_shape": [
    1024,
    1024
  ],
  "o_projection_shape": [
    1024,
    2048
  ],
  "kv_bytes_per_token_fp16_batch1": 114688,
  "vocab_size_from_config": 151936,
  "max_positions_config_not_quality_guarantee": 40960
}
朴素相除： 64
真实配置 head_dim： 128
FP16 KV 每token理论载荷： 112.0 KiB


### KV 载荷的计算仍然适用

固定普通 GQA、未量化 K/V：

\[
\mathrm{bytes/token}=2\times L\times B\times H_{kv}\times d_{head}\times s
\]

用本配置、batch=1、每元素2字节：`2×28×1×8×128×2=114688` 字节，即 **112 KiB/token**。缓存 1024 个位置时仅 K/V 的有效元素约 112 MiB。

这是按公开配置推导的载荷，**不是本机实测显存**。参数、工作区、注意力临时量、分配器空间均不在里面。模型配置的最大位置值也不等于已经验证的长上下文能力；本课另设保守预算 2048，不直接使用上万 token。

## 3. 消息列表不是模型直接接收的 Python 对象

程序可以用：

```python
messages = [
    {"role": "system", "content": "你是一名简洁的中文助手。"},
    {"role": "user", "content": "什么是大语言模型？"},
]
```

但模型实际接收的是 token ID 张量。流程是：

**角色消息 → 配套 Chat Template → 一段文本/控制标记 → token IDs → 模型。**

同一句话，直接编码裸文本与按聊天模板编码，一般不是同一条序列。不同模型的模板也可能不同，不要自己拼一个“看起来差不多”的模板。[S3]

`add_generation_prompt=True` 表示补上“现在轮到 assistant 继续”的模板前缀；它**不是**把模型答案提前填好。不要混用“开始新回答”与“续写未完成 assistant 消息”的模式。

本课先固定 `enable_thinking=False`，控制输出长度和学习范围。这是所选 Qwen3 模板支持的参数，不是所有模型通用的 `generate()` 开关。[S1]

In [6]:
messages = [
    {"role": "system", "content": "你是一名简洁的中文助手。"},
    {"role": "user", "content": "什么是大语言模型？"},
]
print(validate_messages(messages))
try:
    validate_messages([{"role": "tool", "content": "本课尚未支持工具消息"}])
except ValueError as exc:
    print("明确拒绝超出本课范围的输入：", exc)

[{'role': 'system', 'content': '你是一名简洁的中文助手。'}, {'role': 'user', 'content': '什么是大语言模型？'}]
明确拒绝超出本课范围的输入： 不支持这个 role


## 4. 模板生成以后，为什么 `add_special_tokens=False`？

推荐对照两条路线：

```python
# 路线A：模板直接给编号
ids = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, enable_thinking=False
)

# 路线B：先观察完整文本，再编码
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
ids2 = tokenizer(text, add_special_tokens=False)["input_ids"]
```

对相同消息和设置，本课要求两条路线得到相同编号。模板已经负责边界标记，再自动加 BOS/EOS 可能导致重复。[S3]

**不是每个 tokenizer 在 `add_special_tokens=True` 时都会添加东西。** 某个 Qwen 版本两者碰巧相同，也不能据此推广到其他模型。本课离线替身特意选择“默认会添加 BOS”的规则演示风险，绝不冒充真实 Qwen 行为。

In [7]:
def render_chat(tokenizer, messages: list[dict]) -> str:
    clean = validate_messages(messages)
    if not getattr(tokenizer, "chat_template", None):
        raise ValueError("分词器缺少 chat_template；不猜测模板，不使用第27课的教学分词器")
    return tokenizer.apply_chat_template(clean, tokenize=False,
                                         add_generation_prompt=True, enable_thinking=False)


def validate_left_padded_batch(inputs: dict) -> None:
    torch = require_torch()
    if "input_ids" not in inputs or "attention_mask" not in inputs:
        raise ValueError("必须同时传 input_ids 和 attention_mask")
    ids, mask = inputs["input_ids"], inputs["attention_mask"]
    if ids.ndim != 2 or ids.shape != mask.shape or min(ids.shape) <= 0:
        raise ValueError("input_ids 和 attention_mask 必须是相同的非空 [B,T]")
    if ids.dtype != torch.long or torch.any(ids < 0).item():
        raise ValueError("输入编号必须是非负 torch.long，不能把标签的 -100 放进输入")
    if not torch.all((mask == 0) | (mask == 1)).item():
        raise ValueError("attention_mask 只能包含 0/1")
    if not torch.all(mask[:, -1] == 1).item():
        raise ValueError("decoder-only 批量生成采用左补齐，最后位置必须是有效输入")
    if torch.any(mask[:, 1:].long() < mask[:, :-1].long()).item():
        raise ValueError("每行 mask 必须是 0...0,1...1，不能在中间出现洞")


def prepare_chat_batch(tokenizer, conversations: list[list[dict]], *, max_input=512,
                       max_new=64, context_limit=2048) -> tuple[dict, list[str]]:
    if not isinstance(conversations, list) or not conversations:
        raise ValueError("conversations 必须是非空的对话列表")
    if tokenizer.padding_side != "left":
        raise ValueError("先设置 tokenizer.padding_side='left'")
    if tokenizer.pad_token_id is None:
        raise ValueError("配套分词器没有 PAD；不要盲目添加新 token 改变词表")
    rendered = [render_chat(tokenizer, messages) for messages in conversations]
    inputs = tokenizer(rendered, add_special_tokens=False, padding=True,
                       truncation=False, return_tensors="pt", return_attention_mask=True)
    # 只传本课使用的两项，不把任意 tokenizer 字段自动转发给模型。
    inputs = {key: inputs[key] for key in ("input_ids", "attention_mask")}
    validate_left_padded_batch(inputs)
    check_budget(inputs["input_ids"].shape[1], max_input, max_new, context_limit)
    return inputs, rendered


def position_ids_from_mask(mask):
    """用于手工 forward 探测；generate 会自行管理对应的位置。"""
    return (mask.long().cumsum(-1) - 1).masked_fill(mask == 0, 0)


def validate_vocab_contract(tokenizer, embedding_rows: int) -> dict:
    positive_int(embedding_rows, "embedding_rows")
    ids = list(tokenizer.get_vocab().values())
    if not ids or any(type(i) is not int or i < 0 or i >= embedding_rows for i in ids):
        raise ValueError("tokenizer 可产生的 ID 超出模型 Embedding 范围")
    return {"base_vocab_size": tokenizer.vocab_size, "len_tokenizer": len(tokenizer),
            "max_token_id": max(ids), "embedding_rows": embedding_rows,
            "note": "范围检查不是语义兼容证明；仍须使用配套 tokenizer、模板与 revision"}

## 5. 批量生成：左补齐与有效位置掩码

本课采用 decoder-only 的标准 `generate()` 路径。长短不同的提示组成批次时，使用左补齐，让每条序列的最后一列都是有效输入；不能只因为训练时使用右补齐，就原样照搬到生成。[S4]

`attention_mask` 中，1 表示有效 token，0 表示补齐。即使 PAD 和某个 EOS 编号相同，也要显式传 mask，不能只靠 ID 猜测。

本课不静默截断：模板之后的输入超过 512，或输入宽度加 `max_new_tokens` 超过本课预算，会报错。手动整理历史后再重试，而不是隐藏地丢掉 system 或问题前半段。

In [8]:
def extract_new_ids(sequences, input_ids):
    torch = require_torch()
    if sequences.ndim != 2 or input_ids.ndim != 2:
        raise ValueError("这里只支持 [B,T] 张量")
    if sequences.shape[0] != input_ids.shape[0] or sequences.shape[1] < input_ids.shape[1]:
        raise ValueError("生成输出和输入形状不匹配")
    width = input_ids.shape[1]
    if not torch.equal(sequences[:, :width], input_ids):
        raise ValueError("这里假设 decoder-only 输出保留整个 padded 输入前缀")
    # 不能按 attention_mask.sum() 裁剪：generate 输出包含左侧 padding 的整个输入宽度。
    return sequences[:, width:]


def decode_new_ids(tokenizer, new_ids, stop_ids: list[int], max_new: int) -> list[dict]:
    stops = set(eos_ids(stop_ids))
    positive_int(max_new, "max_new")
    result = []
    for row in new_ids.detach().cpu().tolist():
        stop = next((i for i, x in enumerate(row) if x in stops), None)
        if stop is not None:
            kept, reason = row[:stop + 1], "eos"
        else:
            kept = row
            reason = "max_new_tokens" if len(row) >= max_new else "other_stop"
        result.append({"new_token_ids_including_eos": kept,
                       "raw_generated_width": len(row), "stop_reason": reason,
                       "new_tokens_including_eos": len(kept),
                       "raw_text_with_special_tokens": tokenizer.decode(kept, skip_special_tokens=False,
                                                                          clean_up_tokenization_spaces=False),
                       "answer": tokenizer.decode(kept, skip_special_tokens=True,
                                                  clean_up_tokenization_spaces=False)})
    return result


def generation_options(stop_ids, pad_id, max_new=64, sample=False,
                       temperature=0.7, top_p=0.8, top_k=20) -> dict:
    positive_int(max_new, "max_new")
    if type(pad_id) is not int or pad_id < 0 or type(sample) is not bool:
        raise ValueError("PAD 必须是合法 ID，sample 必须是 bool")
    options = {"max_new_tokens": max_new, "do_sample": sample, "num_beams": 1,
               "use_cache": True, "eos_token_id": eos_ids(stop_ids), "pad_token_id": pad_id,
               "return_dict_in_generate": True, "output_scores": False}
    if sample:
        if (isinstance(temperature, bool) or not isinstance(temperature, (int, float))
                or not math.isfinite(temperature) or temperature <= 0):
            raise ValueError("采样温度必须为正的有限数")
        if isinstance(top_p, bool) or not isinstance(top_p, (int, float)) or not 0 < top_p <= 1:
            raise ValueError("top_p 必须在 (0,1]")
        positive_int(top_k, "top_k")
        options.update(temperature=float(temperature), top_p=float(top_p), top_k=top_k)
    # greedy 时不夹带温度/top-p，避免混淆实际生效规则。
    return options

In [9]:
# 这个例子只有张量，不依赖真实 tokenizer。
import torch
input_ids = torch.tensor([[0, 0, 10, 11], [20, 21, 22, 23]])
mask = torch.tensor([[0, 0, 1, 1], [1, 1, 1, 1]])
sequences = torch.tensor([[0, 0, 10, 11, 40, 2], [20, 21, 22, 23, 50, 2]])
print("补齐后的公共宽度：", input_ids.shape[1])
print("各自有效长度：", mask.sum(-1).tolist())
print("正确只取新增：", extract_new_ids(sequences, input_ids).tolist())
print("错误按有效长度切：", sequences[0, int(mask[0].sum()):].tolist())

补齐后的公共宽度： 4
各自有效长度： [2, 4]
正确只取新增： [[40, 2], [50, 2]]
错误按有效长度切： [10, 11, 40, 2]


## 6. 为什么不能按每条提示的有效长度截取回答？

对于本课的 decoder-only `generate()`，返回的 `sequences` 包含**整个已经补齐的输入**，随后才是新增内容。[S4]

上例的短提示实际只有2个有效 token，但返回值仍以4列输入开头。因此：

```python
new_ids = sequences[:, input_ids.shape[1]:]
```

才是正确的切法。用 `attention_mask.sum()` 切片，会把短提示本身混进回答。

EOS 后的批量补齐不应计入有效生成长度。本课读取 `generation_config` 中的全部 EOS，而不是只取 tokenizer 的一个 EOS。所选仓库的生成配置定义了不止一个结束 ID。[S8]

`max_new_tokens` 限制新增数量，不是输入加输出的总长；没有 EOS 而达到预算时，报告 `max_new_tokens`，不把被截断的句子伪装成正常完成。[S5]

In [10]:
print("贪心参数：", generation_options([151645, 151643], 151643, max_new=32))
print("采样参数：", generation_options([151645, 151643], 151643, max_new=32, sample=True))
check_budget(input_width=80, max_input=512, max_new=64, context_limit=2048)
print("80 + 64 的预算检查通过")

贪心参数： {'max_new_tokens': 32, 'do_sample': False, 'num_beams': 1, 'use_cache': True, 'eos_token_id': [151645, 151643], 'pad_token_id': 151643, 'return_dict_in_generate': True, 'output_scores': False}
采样参数： {'max_new_tokens': 32, 'do_sample': True, 'num_beams': 1, 'use_cache': True, 'eos_token_id': [151645, 151643], 'pad_token_id': 151643, 'return_dict_in_generate': True, 'output_scores': False, 'temperature': 0.7, 'top_p': 0.8, 'top_k': 20}
80 + 64 的预算检查通过


## 7. 工程实现：受控加载与真实配置检查

下面是可复用实现。阅读时重点看：

1. `revision` 同时传给 config、tokenizer、model，且不执行自定义远程 Python。
2. 用 `dtype` 选择权重载入类型，但不把整数 token ID 转成浮点数。
3. 不使用 `device_map="auto"`，本课显式单设备，没有隐式 offload，也不依赖 Accelerate。
4. tokenizer 的 `len()`、基础词表大小和模型 Embedding 行数可能不同；只看数量相同不是兼容证明。[S6]

本课只支持所选 Qwen3 文本模型，不假装这些参数可以无修改适配所有公开模型。

In [11]:
def choose_runtime(device: str, dtype: str):
    torch = require_torch()
    if device not in {"auto", "cpu", "cuda"}:
        raise ValueError("本课仅支持 cpu、cuda 或 auto 单设备")
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("明确指定了 CUDA，但当前 PyTorch 不可用；不会悄悄回退到 CPU")
    if dtype == "auto":
        dtype = "float32" if device == "cpu" else (
            "bfloat16" if torch.cuda.is_bf16_supported() else "float16")
    if dtype not in {"float32", "float16", "bfloat16"}:
        raise ValueError("dtype 必须是 auto/float32/float16/bfloat16")
    if device == "cpu" and dtype == "float16":
        raise ValueError("本课不使用 CPU FP16；默认使用 CPU FP32")
    if device == "cuda" and dtype == "bfloat16" and not torch.cuda.is_bf16_supported():
        raise RuntimeError("当前 CUDA 环境未报告 BF16 支持，改用 float16 或 float32")
    return torch.device(device), getattr(torch, dtype)


def source_options(model_source: str, revision: str | None, local_files_only: bool) -> tuple[str, dict]:
    path = Path(model_source).expanduser()
    if path.is_dir():
        if not (path / "config.json").is_file():
            raise ValueError("本地模型目录缺少 config.json")
        return str(path.resolve()), {"local_files_only": True, "trust_remote_code": False}
    if model_source != MODEL_ID:
        raise ValueError("本课只接受 Qwen/Qwen3-0.6B 或已有的本地 Qwen3 模型目录")
    return model_source, {"revision": revision or REVISION, "local_files_only": local_files_only,
                          "trust_remote_code": False}


def inspect_tokenizer(model_source=MODEL_ID, revision=None, local_files_only=False):
    AutoConfig, AutoTokenizer, _, _ = require_transformers()
    source, options = source_options(model_source, revision, local_files_only)
    config = AutoConfig.from_pretrained(source, **options)
    if config.model_type != "qwen3":
        raise ValueError("本课模型实现限定为 qwen3，不猜测其他架构和 thinking 参数")
    tokenizer = AutoTokenizer.from_pretrained(source, padding_side="left", **options)
    if not isinstance(tokenizer.chat_template, str) or "enable_thinking" not in tokenizer.chat_template:
        raise ValueError("本课需要支持 enable_thinking 的配套 Qwen3 模板")
    messages = [{"role": "user", "content": "你好，请简短回答。"}]
    text = render_chat(tokenizer, messages)
    direct = tokenizer.apply_chat_template(messages, tokenize=True,
                add_generation_prompt=True, enable_thinking=False)
    two_stage = tokenizer(text, add_special_tokens=False)["input_ids"]
    if direct != two_stage:
        raise RuntimeError("模板直接编码与文本后编码不一致；停止，不猜测修复")
    plain = tokenizer(messages[0]["content"], add_special_tokens=False)["input_ids"]
    no_header = tokenizer.apply_chat_template(messages, tokenize=False,
                add_generation_prompt=False, enable_thinking=False)
    thinking_text = tokenizer.apply_chat_template(messages, tokenize=False,
                add_generation_prompt=True, enable_thinking=True)
    report = {
        "source": source, "requested_revision": options.get("revision"),
        "resolved_config_commit": getattr(config, "_commit_hash", None),
        "architecture": architecture_summary(config.to_dict()),
        "vocab_contract": validate_vocab_contract(tokenizer, config.vocab_size),
        "template_sha256": hashlib.sha256(tokenizer.chat_template.encode()).hexdigest(),
        "plain_token_count": len(plain), "chat_token_count": len(direct),
        "chat_ids": direct, "rendered_text": text,
        "without_generation_prompt": no_header,
        "thinking_template_only_not_generation": thinking_text,
        "two_tokenization_routes_equal": direct == two_stage,
        "pad_token_id": tokenizer.pad_token_id, "tokenizer_eos_id": tokenizer.eos_token_id,
        "tokenizer_bos_id": tokenizer.bos_token_id,
        "special_tokens_map": tokenizer.special_tokens_map,
        "config_fingerprint": fingerprint(config.to_dict()),
        "note": "只下载配置/分词器；没有加载模型权重，也没有产生回答。",
    }
    return config, tokenizer, report, source, options


def summarize_model(model, tokenizer) -> dict:
    parameters = list(model.parameters())  # parameters() 会去重共享 Parameter。
    embedding = model.get_input_embeddings().weight
    head = model.get_output_embeddings().weight
    contract = validate_vocab_contract(tokenizer, embedding.shape[0])
    info = {"unique_parameter_count": sum(p.numel() for p in parameters),
            "parameter_payload_bytes": sum(p.numel() * p.element_size() for p in parameters),
            "parameter_dtypes": sorted({str(p.dtype) for p in parameters}),
            "parameter_devices": sorted({str(p.device) for p in parameters}),
            "embedding_shape": list(embedding.shape), "lm_head_shape": list(head.shape),
            "tied_embedding_storage": embedding.data_ptr() == head.data_ptr(),
            "vocab_contract": contract,
            "memory_note": "参数有效载荷不等于权重文件大小、加载峰值或推理峰值。"}
    first_attention = model.model.layers[0].self_attn
    info["first_layer_projection_shapes"] = {
        name: list(getattr(first_attention, name).weight.shape)
        for name in ("q_proj", "k_proj", "v_proj", "o_proj")}
    info["first_layer_has_qk_norm"] = all(hasattr(first_attention, name) for name in ("q_norm", "k_norm"))
    return info

## 8. 工程实现：真正的生成与前向观察

生成函数执行 `eval()` 和 `inference_mode()`，不调用优化器；返回之后恢复原模块模式和随机状态。它会记录消息、模板、输入 ID、mask、全部有效新增 ID、停止原因和生效的生成配置。

单次 `generate` 的墙钟包括 prefill 和 decode，不包括下载、模型加载和分词；**不能把这个数字叫作 GPU 内核速度或 TTFT**。本课也不声称单次计时是稳定基准。

还提供一次手工前向：检查 logits 的形状 `[B,T,V]`，查看最后一个位置的前5候选。单个 ID 的显示可能是不完整字节片段；最终回答要对整段新增 ID 一起解码，继续沿用第27课的认识。

默认贪心用于简化接口复现，**不是声称最优质量设置**。模型卡为非 thinking 模式推荐采样温度0.7、top-p 0.8、top-k 20；本课 `--sample` 使用这些设置。思考模式有另外的建议，本课只查看对应模板，不进行思考模式生成。[S1]

In [12]:
def generate_batch(model, tokenizer, conversations: list[list[dict]], *, device="cpu",
                   max_input=512, max_new=64, context_limit=2048, sample=False, seed=28,
                   generation_config_factory=None) -> dict:
    torch = require_torch()
    if type(seed) is not int or seed < 0:
        raise ValueError("seed 必须是非负整数")
    if generation_config_factory is None:
        _, _, _, generation_config_factory = require_transformers()
    model_limit = int(model.config.max_position_embeddings)
    effective_limit = min(positive_int(context_limit, "context_limit"), model_limit)
    inputs, rendered = prepare_chat_batch(tokenizer, conversations, max_input=max_input,
        max_new=max_new, context_limit=effective_limit)
    device = torch.device(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    stopping = getattr(model.generation_config, "eos_token_id", None)
    if stopping is None:
        stopping = tokenizer.eos_token_id
    stops = eos_ids(stopping)
    options = generation_options(stops, tokenizer.pad_token_id, max_new=max_new, sample=sample)
    # 新建配置，避免把仓库里 do_sample、beam、重复惩罚等默认值误当成本课设置。
    gen_config = generation_config_factory(**options)
    modes = [(module, module.training) for module in model.modules()]
    cuda_devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
    try:
        model.eval()
        # 恢复所有 CUDA RNG，避免 torch.manual_seed 改动其他设备的状态。
        with torch.random.fork_rng(devices=cuda_devices), torch.inference_mode():
            torch.manual_seed(seed)
            if device.type == "cuda":
                torch.cuda.synchronize(device)
            start = time.perf_counter()
            output = model.generate(**inputs, generation_config=gen_config)
            if device.type == "cuda":
                torch.cuda.synchronize(device)
            elapsed = time.perf_counter() - start
        generated = extract_new_ids(output.sequences, inputs["input_ids"])
        replies = decode_new_ids(tokenizer, generated, stops, max_new)
    finally:
        for module, was_training in modes:
            module.training = was_training
    return {"messages": copy.deepcopy(conversations), "rendered_prompts": rendered,
            "input_ids": inputs["input_ids"].cpu().tolist(),
            "attention_mask": inputs["attention_mask"].cpu().tolist(),
            "input_shape": list(inputs["input_ids"].shape),
            "valid_prompt_lengths": inputs["attention_mask"].sum(-1).cpu().tolist(),
            "generation_config": options, "seed": seed, "responses": replies,
            "generate_wall_seconds": elapsed,
            "timing_note": "单次 generate 墙钟，包含 prefill+decode；不含下载/加载/分词，不是稳定性能基准或 TTFT。",
            "scope": "批量续写，不更新参数；未传入的历史不会自动记住。"}


def inspect_forward(model, tokenizer, messages, device, max_input=512) -> dict:
    torch = require_torch()
    inputs, _ = prepare_chat_batch(tokenizer, [messages], max_input=max_input, max_new=1,
                                  context_limit=min(2048, model.config.max_position_embeddings))
    inputs = {k: v.to(device) for k, v in inputs.items()}
    was_training = model.training
    try:
        model.eval()
        with torch.inference_mode():
            out = model(**inputs, position_ids=position_ids_from_mask(inputs["attention_mask"]),
                        use_cache=False, return_dict=True)
            logits = out.logits
            if not torch.isfinite(logits).all().item():
                raise RuntimeError("前向 logits 出现非有限数值")
            probabilities = logits[0, -1].float().softmax(-1)
            values, indices = torch.topk(probabilities, k=5)
            candidates = [{"token_id": i, "probability": p,
                           "token_piece_not_independent_character": tokenizer.convert_ids_to_tokens(i),
                           "single_id_display_may_be_lossy": tokenizer.decode([i], skip_special_tokens=False,
                                                                             clean_up_tokenization_spaces=False)}
                          for i, p in zip(indices.cpu().tolist(), values.cpu().tolist())]
            return {"input_shape": list(inputs["input_ids"].shape), "logits_shape": list(logits.shape),
                    "top5_at_last_input_position": candidates,
                    "note": "单 token 显示可能是不完整字符；最终答案始终整体 decode。"}
    finally:
        model.train(was_training)

In [13]:
@dataclass
class RunSettings:
    model: str = MODEL_ID
    revision: str | None = None
    device: str = "auto"
    dtype: str = "auto"
    local_files_only: bool = False
    max_input: int = 512
    max_new: int = 64
    context_limit: int = 2048
    sample: bool = False
    seed: int = 28
    batch_demo: bool = False
    output_dir: str = "lesson_28_outputs"


def run_real(settings: RunSettings, messages: list[dict], *, inspect_only=False) -> dict:
    validate_messages(messages)
    # 防止把教学输出写入用户的模型源目录。
    source_path = Path(settings.model).expanduser()
    output_root = Path(settings.output_dir).expanduser().resolve()
    if source_path.is_dir() and output_root.is_relative_to(source_path.resolve()):
        raise ValueError("输出目录不能位于输入模型目录内")
    run_dir = output_root / datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%S_%fZ")
    run_dir.mkdir(parents=True, exist_ok=False)
    manifest = {"format": FORMAT, "status": "started", "settings": asdict(settings),
                "environment": environment_info(), "sources": SOURCES,
                "inspect_only": inspect_only, "privacy": "报告保存输入/输出明文，请勿放入密钥或敏感资料"}
    try:
        json_write(run_dir / "manifest.json", manifest)
        config, tokenizer, inspection, source, options = inspect_tokenizer(
            settings.model, settings.revision, settings.local_files_only)
        json_write(run_dir / "inspection.json", inspection)
        if inspect_only:
            manifest.update(status="success", weights_loaded=False)
        else:
            torch = require_torch()
            _, _, AutoModelForCausalLM, _ = require_transformers()
            device, dtype = choose_runtime(settings.device, settings.dtype)
            # 单设备，故意不使用 device_map='auto'；不需要 accelerate，不发生隐式分层 offload。
            model = AutoModelForCausalLM.from_pretrained(source, dtype=dtype,
                attn_implementation="sdpa", use_safetensors=True, **options).to(device)
            model.eval()
            model_info = summarize_model(model, tokenizer)
            json_write(run_dir / "model.json", model_info)
            forward = inspect_forward(model, tokenizer, messages, device, settings.max_input)
            json_write(run_dir / "forward.json", forward)
            conversations = [messages]
            if settings.batch_demo:
                conversations.append([{"role": "user", "content": "只回答一个整数：2+3等于几？"}])
            report = generate_batch(model, tokenizer, conversations, device=device,
                max_input=settings.max_input, max_new=settings.max_new, context_limit=settings.context_limit,
                sample=settings.sample, seed=settings.seed)
            json_write(run_dir / "generations.json", report)
            manifest.update(status="success", weights_loaded=True,
                actual_device=str(device), actual_dtype=str(dtype),
                note="本机真实模型结果；不是自动通过的能力基准，不执行回答中的代码。")
            for i, response in enumerate(report["responses"], 1):
                print(f"\n回答 {i}（停止原因：{response['stop_reason']}）：\n{response['answer']}")
            del model
        json_write(run_dir / "manifest.json", manifest)
    except Exception as exc:
        manifest.update(status="failed", error_type=type(exc).__name__, error=str(exc))
        json_write(run_dir / "manifest.json", manifest)
        print(f"本次失败记录：{run_dir / 'manifest.json'}", file=sys.stderr)
        raise
    print(f"本次结果目录：{run_dir}")
    return {"run_dir": str(run_dir), "manifest": manifest, "inspection": inspection}

## 9. 离线实验：教学替身只验证程序约定

接下来定义一个字节 tokenizer 和固定返回“好”的模型替身。它们完全不是 Qwen，不会学习或推理。

它们的用途是：没有网络和权重时，仍然能检查自己的包装程序有没有重复添加边界、用错 padding、裁剪错回答、忘记恢复模式等问题。

**替身测试通过，不证明 Transformers 接口兼容，更不证明真实模型回答正确。**

In [14]:
class FixtureTokenizer:
    """UTF-8 字节测试替身。PAD=0,BOS=1,EOS=2；字节ID=byte+3。不是生产 tokenizer。"""
    padding_side = "left"
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    chat_template = "OFFLINE FIXTURE ONLY; enable_thinking"
    vocab_size = 259

    def __len__(self):
        return 259

    def get_vocab(self):
        return {str(i): i for i in range(259)}

    def apply_chat_template(self, messages, tokenize=False, add_generation_prompt=False,
                           enable_thinking=False):
        text = "".join(f"<{m['role']}>\n{m['content']}\n</{m['role']}>\n" for m in messages)
        if add_generation_prompt:
            text += "<assistant>\n"
            if not enable_thinking:
                text += "<fixture_no_think>\n"
        return self(text, add_special_tokens=False)["input_ids"] if tokenize else text

    def __call__(self, text, add_special_tokens=True, padding=False, truncation=False,
                 return_tensors=None, return_attention_mask=True):
        self.last_call = {"add_special_tokens": add_special_tokens, "padding": padding,
                          "truncation": truncation}
        strings = [text] if isinstance(text, str) else text
        rows = [[x + 3 for x in item.encode("utf-8")] for item in strings]
        if add_special_tokens:
            rows = [[1] + row for row in rows]
        if return_tensors is None:
            return {"input_ids": rows[0] if isinstance(text, str) else rows}
        torch = require_torch()
        width = max(map(len, rows))
        ids, masks = [], []
        for row in rows:
            delta = width - len(row)
            if self.padding_side == "left":
                ids.append([0] * delta + row); masks.append([0] * delta + [1] * len(row))
            else:
                ids.append(row + [0] * delta); masks.append([1] * len(row) + [0] * delta)
        return {"input_ids": torch.tensor(ids, dtype=torch.long),
                "attention_mask": torch.tensor(masks, dtype=torch.long)}

    def decode(self, ids, skip_special_tokens=True, clean_up_tokenization_spaces=False):
        data, parts = [], []
        for i in ids:
            if i < 3:
                if data:
                    parts.append(bytes(data).decode("utf-8", errors="replace")); data = []
                if not skip_special_tokens:
                    parts.append({0: "<pad>", 1: "<bos>", 2: "<eos>"}[i])
            else:
                data.append(i - 3)
        if data:
            parts.append(bytes(data).decode("utf-8", errors="replace"))
        return "".join(parts)


def make_fixture_model(fail=False):
    torch = require_torch()

    class FixtureModel(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.weight = torch.nn.Parameter(torch.tensor([1.0]))
            self.dropout = torch.nn.Dropout(0.1)
            self.config = SimpleNamespace(max_position_embeddings=2048)
            self.generation_config = SimpleNamespace(eos_token_id=[2, 0])

        def generate(self, input_ids, attention_mask, generation_config):
            if fail:
                raise RuntimeError("故意模拟生成失败，不是模型回答")
            if self.training or torch.is_grad_enabled():
                raise RuntimeError("生成应处于 eval 和无梯度模式")
            validate_left_padded_batch({"input_ids": input_ids, "attention_mask": attention_mask})
            # 故意消耗 RNG，用于检查外层是否恢复，不用于模拟真实模型。
            torch.rand(3)
            answer = [x + 3 for x in "好".encode()] + [2]
            answer = answer[:generation_config.max_new_tokens]
            suffix = torch.tensor([answer] * input_ids.shape[0], device=input_ids.device)
            return SimpleNamespace(sequences=torch.cat((input_ids, suffix), dim=1))

    return FixtureModel()

In [15]:
fixture_tokenizer = FixtureTokenizer()
fixture_model = make_fixture_model()
fixture_conversations = [
    [{"role": "user", "content": "你好"}],
    [{"role": "user", "content": "请解释一个更长的问题。"}],
]
prepared, rendered = prepare_chat_batch(fixture_tokenizer, fixture_conversations)
print("以下是教学替身模板，不是真实 Qwen：")
print(rendered[0])
print("输入形状：", tuple(prepared['input_ids'].shape))
print("各行有效长度：", prepared['attention_mask'].sum(-1).tolist())
print("左侧补齐：", prepared['attention_mask'][0, :10].tolist())

以下是教学替身模板，不是真实 Qwen：
<user>
你好
</user>
<assistant>
<fixture_no_think>

输入形状： (2, 80)
各行有效长度： [53, 80]
左侧补齐： [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [16]:
before = fixture_model.weight.detach().clone()
report_fixture = generate_batch(
    fixture_model, fixture_tokenizer, fixture_conversations,
    max_new=8, generation_config_factory=SimpleNamespace,
)
print("教学替身结果，不是模型实测：")
for row in report_fixture['responses']:
    print(row['answer'], row['stop_reason'], row['new_token_ids_including_eos'])
print("测试参数未变化：", torch.equal(before, fixture_model.weight))
print("输入消息未添加回答：", fixture_conversations[0])

教学替身结果，不是模型实测：
好 eos [232, 168, 192, 2]
好 eos [232, 168, 192, 2]
测试参数未变化： True
输入消息未添加回答： [{'role': 'user', 'content': '你好'}]


## 10. 完整自动检查

测试涵盖消息角色、上下文预算、显式 head_dim、词表范围、padding、EOS、Unicode整段解码、生成模式与RNG恢复、报错恢复、来源和配置记录。

这些检查**不需要 Transformers，不进行网络请求**。不能把下面的通过数量写成“真实 Qwen 已通过多少项测试”。

In [17]:
def run_checks(verbose=True) -> dict:
    torch = require_torch()
    names = []

    def check(name, condition):
        if not bool(condition):
            raise AssertionError(name)
        names.append(name)

    def raises(name, fn, exc=Exception):
        try:
            fn()
        except exc:
            names.append(name)
        else:
            raise AssertionError(f"应该拒绝：{name}")

    msg = [{"role": "user", "content": "你好"}]
    original = copy.deepcopy(msg)
    check("合法消息复制", validate_messages(msg) == msg and validate_messages(msg) is not msg)
    check("系统消息", len(validate_messages([{"role": "system", "content": "助手"}] + msg)) == 2)
    for name, bad in [("空对话", []), ("结尾assistant", [{"role": "assistant", "content": "a"}]),
                      ("重复user", msg + msg), ("工具消息", [{"role": "tool", "content": "x"}]),
                      ("非文本", [{"role": "user", "content": ["x"]}]),
                      ("额外字段", [{"role": "user", "content": "x", "name": "x"}])]:
        raises(name, lambda bad=bad: validate_messages(bad), (ValueError, TypeError))
    history = extend_conversation(msg, "回答", "追问")
    check("完整历史", [m["role"] for m in history] == ["user", "assistant", "user"])
    history[0]["content"] = "changed"
    check("历史不改输入", msg == original)
    raises("追问必须文本", lambda: extend_conversation(msg, "a", 3), TypeError)
    check("多EOS去重", eos_ids([2, 3, 2]) == [2, 3])
    check("单EOS", eos_ids(2) == [2])
    for bad in (None, [], True, [-1], [False]):
        raises(f"非法EOS {bad}", lambda bad=bad: eos_ids(bad), ValueError)
    check_budget(10, 10, 2, 12); check("预算等于边界", True)
    raises("预算不截断", lambda: check_budget(11, 10, 1, 20), ValueError)
    raises("保留输出预算", lambda: check_budget(10, 20, 3, 12), ValueError)
    raises("布尔不是长度", lambda: check_budget(True, 10, 2, 20), ValueError)
    arch = architecture_summary(CONFIG_SNAPSHOT)
    check("显式head_dim", arch["head_dim"] == 128)
    check("Q投影宽度", arch["q_projection_shape"] == [2048, 1024])
    check("O投影宽度", arch["o_projection_shape"] == [1024, 2048])
    check("KV估算", arch["kv_bytes_per_token_fp16_batch1"] == 114688)
    fallback = dict(CONFIG_SNAPSHOT); fallback.pop("head_dim")
    check("缺省head_dim回退", architecture_summary(fallback)["head_dim"] == 64)
    broken = dict(CONFIG_SNAPSHOT, num_key_value_heads=3)
    raises("非法GQA", lambda: architecture_summary(broken), ValueError)
    tok = FixtureTokenizer()
    conversations = [msg, [{"role": "user", "content": "请解释一个较长的问题。"}]]
    inputs, texts = prepare_chat_batch(tok, conversations, max_input=512)
    check("special tokens关闭", tok.last_call["add_special_tokens"] is False)
    check("不自动截断", tok.last_call["truncation"] is False)
    check("左补齐", inputs["attention_mask"][0, 0].item() == 0)
    check("末位置有效", bool(torch.all(inputs["attention_mask"][:, -1] == 1)))
    check("assistant开始", texts[0].endswith("<assistant>\n<fixture_no_think>\n"))
    check("输入包含消息文本", "你好" in texts[0])
    direct = tok.apply_chat_template(msg, tokenize=True, add_generation_prompt=True, enable_thinking=False)
    check("两条编码路线一致", direct == tok(texts[0], add_special_tokens=False)["input_ids"])
    check("默认加BOS的教学反例", tok(texts[0])["input_ids"] != direct)
    tok.padding_side = "right"
    raises("拒绝右补齐设置", lambda: prepare_chat_batch(tok, conversations), ValueError)
    tok.padding_side = "left"
    raises("模板后超长", lambda: prepare_chat_batch(tok, conversations, max_input=2), ValueError)
    bad = {"input_ids": torch.ones(1, 3, dtype=torch.long), "attention_mask": torch.tensor([[1, 1, 0]])}
    raises("右侧PAD掩码", lambda: validate_left_padded_batch(bad), ValueError)
    bad["attention_mask"] = torch.tensor([[1, 0, 1]])
    raises("中间mask空洞", lambda: validate_left_padded_batch(bad), ValueError)
    bad["attention_mask"] = torch.tensor([[0, 0, 0]])
    raises("全PAD", lambda: validate_left_padded_batch(bad), ValueError)
    pos = position_ids_from_mask(torch.tensor([[0, 0, 1, 1], [1, 1, 1, 1]]))
    check("补齐后位置", pos.tolist() == [[0, 0, 0, 1], [0, 1, 2, 3]])
    check("词表范围不要求相等", validate_vocab_contract(tok, 264)["embedding_rows"] == 264)
    raises("词表越界", lambda: validate_vocab_contract(tok, 258), ValueError)
    x = torch.tensor([[0, 0, 5, 6], [5, 6, 7, 8]])
    y = torch.cat((x, torch.tensor([[100, 2, 0], [101, 102, 2]])), dim=1)
    new = extract_new_ids(y, x)
    check("按统一宽度切片", new.tolist() == [[100, 2, 0], [101, 102, 2]])
    check("按有效长度切片是错的", y[0, 2:].tolist() != new[0].tolist())
    broken_y = y.clone(); broken_y[0, 2] = 99
    raises("生成前缀被改写", lambda: extract_new_ids(broken_y, x), ValueError)
    decoded = decode_new_ids(tok, new, [2, 0], 3)
    check("EOS后padding不计数", decoded[0]["new_tokens_including_eos"] == 2)
    check("两条都EOS", all(r["stop_reason"] == "eos" for r in decoded))
    length_only = decode_new_ids(tok, torch.tensor([[100, 101]]), [2], 2)
    check("长度停止", length_only[0]["stop_reason"] == "max_new_tokens")
    unicode_ids = torch.tensor([[x + 3 for x in "中".encode()] + [2]])
    check("整段解码UTF8", decode_new_ids(tok, unicode_ids, [2], 4)[0]["answer"] == "中")
    g = generation_options([2, 0], 0)
    check("不继承仓库采样", not g["do_sample"] and "temperature" not in g)
    check("保留多个EOS", g["eos_token_id"] == [2, 0])
    check("不保存每步全词表分数", g["output_scores"] is False)
    s = generation_options([2], 0, sample=True)
    check("采样规则明确", s["temperature"] == 0.7 and s["top_p"] == 0.8 and s["top_k"] == 20)
    raises("温度零", lambda: generation_options([2], 0, sample=True, temperature=0), ValueError)
    raises("top_p越界", lambda: generation_options([2], 0, sample=True, top_p=1.1), ValueError)
    device, dtype = choose_runtime("cpu", "auto")
    check("CPU默认FP32", str(device) == "cpu" and dtype == torch.float32)
    raises("CPU不静默FP16", lambda: choose_runtime("cpu", "float16"), ValueError)
    if not torch.cuda.is_available():
        raises("CUDA不静默回退", lambda: choose_runtime("cuda", "auto"), RuntimeError)
    model = make_fixture_model()
    model.train(); model.dropout.eval()
    modes = [m.training for m in model.modules()]
    weight = model.weight.detach().clone()
    model.weight.grad = torch.tensor([7.0])
    rng = torch.get_rng_state().clone()
    report = generate_batch(model, tok, conversations, generation_config_factory=SimpleNamespace, max_new=8)
    check("替身回答，仅测试", all(x["answer"] == "好" for x in report["responses"]))
    check("生成恢复所有模块模式", modes == [m.training for m in model.modules()])
    check("生成恢复RNG", torch.equal(rng, torch.get_rng_state()))
    check("生成不改测试参数", torch.equal(weight, model.weight))
    check("生成不清已有梯度", model.weight.grad.item() == 7)
    failing = make_fixture_model(fail=True); failing.train()
    rng = torch.get_rng_state().clone()
    raises("异常不吞掉", lambda: generate_batch(failing, tok, [msg], generation_config_factory=SimpleNamespace), RuntimeError)
    check("异常恢复模型模式", failing.training)
    check("异常恢复RNG", torch.equal(rng, torch.get_rng_state()))
    check("原消息保持", msg == original)
    check("配置指纹确定", fingerprint({"a": 1, "b": 2}) == fingerprint({"b": 2, "a": 1}))
    with tempfile.TemporaryDirectory() as temp:
        path = Path(temp) / "r.json"; json_write(path, {"文字": "中", "结果": report})
        check("报告可序列化", json.loads(path.read_text(encoding="utf-8"))["文字"] == "中")
        (Path(temp) / "config.json").write_text("{}", encoding="utf-8")
        _, opts = source_options(temp, None, False)
        check("本地源不联网", opts["local_files_only"] and not opts["trust_remote_code"])
    _, opts = source_options(MODEL_ID, None, False)
    check("默认固定commit", opts["revision"] == REVISION and len(REVISION) == 40)
    check("不执行远程模型代码", opts["trust_remote_code"] is False)
    result = {"passed": len(names), "checks": names,
              "scope": "离线逻辑与显式教学替身；未加载真实 Qwen tokenizer/权重，未验证 Transformers 集成或 GPU。"}
    if verbose:
        print(f"离线检查通过：{len(names)} 项。")
        print(result["scope"])
    return result

In [18]:
offline_checks = run_checks()
print("实际运行的检查数：", offline_checks['passed'])

离线检查通过：74 项。
离线逻辑与显式教学替身；未加载真实 Qwen tokenizer/权重，未验证 Transformers 集成或 GPU。
实际运行的检查数： 74


## 11. 现在到你的电脑：先只加载真实分词器

确认依赖安装好，设置 `RUN_REAL_INSPECTION=True` 后运行下一个单元。首次需要联网下载 config 和 tokenizer，但还不会加载模型权重。

真实检查将保存并显示：

- 配置中的投影形状与模板哈希；
- 原文和聊天输入的 token 数；
- `add_generation_prompt=True/False` 的区别；
- `enable_thinking=True/False` 的模板差别；
- 模板直接编码与分两步编码是否相同。

**注意看真实模板中可能出现的空 `<think>...` 前缀。** “禁用 thinking”并不意味着模板文本一定没有这个字符串；这不是模型额外生成了思考内容，而是模板对模式的编码。[S8]

In [19]:
RUN_REAL_INSPECTION = False
RUN_REAL_GENERATION = False
RUN_BATCH_DEMO = False
LOCAL_FILES_ONLY = False  # 首次需要 False；缓存完整后才改 True
DEVICE = "auto"          # 确定测试3060时改 "cuda"，不允许静默CPU回退
DTYPE = "auto"
MODEL_SOURCE = MODEL_ID  # 也可改为自己已准备好的本地 Qwen3 模型目录
MAX_NEW = 64

real_messages = [{"role": "user", "content": "请用一句中文解释什么是大语言模型。"}]
settings = RunSettings(model=MODEL_SOURCE, device=DEVICE, dtype=DTYPE,
                       local_files_only=LOCAL_FILES_ONLY, max_new=MAX_NEW,
                       batch_demo=RUN_BATCH_DEMO)

In [20]:
if RUN_REAL_INSPECTION:
    inspected = run_real(settings, real_messages, inspect_only=True)
    print(json.dumps(inspected['inspection'], ensure_ascii=False, indent=2))
else:
    print("未执行真实分词器加载。将 RUN_REAL_INSPECTION=True 后运行；此处没有真实 token IDs 输出。")

未执行真实分词器加载。将 RUN_REAL_INSPECTION=True 后运行；此处没有真实 token IDs 输出。


## 12. 加载真实权重并生成

把 `RUN_REAL_GENERATION=True` 后执行下一单元。代码会加载权重到一个设备，先做一次前向形状检查，再调用生成。

默认 `max_new_tokens=64`，停止原因若是 `max_new_tokens`，说明输出预算用完，不意味着自然结束。不要为了“看起来完整”在报告中隐藏这一信息。

真实结果写到新建的时间戳目录，不用旧目录残留的成功回答伪装成本次成功。失败也会写 `manifest.json`，记录版本与异常。

报告包含输入/输出明文。不要将密码、API密钥或私人资料放进教学提示；若分享报告先删除敏感内容。下载阶段向仓库请求文件，模型前向在本地运行，本课没有调用在线推理 API。

In [21]:
if RUN_REAL_GENERATION:
    actual = run_real(settings, real_messages)
    print("实际结果目录：", actual['run_dir'])
    print("实际设备：", actual['manifest']['actual_device'])
    print("实际 dtype：", actual['manifest']['actual_dtype'])
else:
    print("未加载真实权重，未生成真实回答。把开关设为 True 后在本机运行。")

未加载真实权重，未生成真实回答。把开关设为 True 后在本机运行。


### 最短调用链：与完整实现对照

下面只是便于阅读的骨架，不替代上面的预算、日志与错误检查。不要把它粘贴后又同时执行另一份模型加载。

```python
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, dtype=torch.float32,
    trust_remote_code=False, use_safetensors=True,
)
text = tokenizer.apply_chat_template(
    real_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
)
inputs = tokenizer([text], add_special_tokens=False, return_tensors="pt")
model.eval()
with torch.inference_mode():
    generated = model.generate(**inputs, max_new_tokens=64, do_sample=False)
new_ids = generated[:, inputs["input_ids"].shape[1]:]
answer = tokenizer.batch_decode(new_ids, skip_special_tokens=True)
```

完整实现另外新建 GenerationConfig，避免仓库中的采样默认项和贪心设置混在一起；骨架仅展示主要调用关系。

## 13. 多轮对话不是模型永久记忆

第一次调用后，程序需要把 assistant 的回答加入 history，再追加 user 的新问题，然后重新套模板。

调用 `generate()` 不会自动保存一份聊天记录。历史传得越多，输入越长，预算也越紧。这个变化不是权重更新。[S3]

本课每次重新构建输入，不跨调用复用 past_key_values；持久会话与缓存复用将在后续工程章节单独处理。

In [22]:
first_history = [{"role": "user", "content": "我正在学习大模型。"}]
second_history = extend_conversation(first_history, "我们可以从分词器开始。", "下一步学习什么？")
print("原消息数：", len(first_history))
print("下一轮消息：", json.dumps(second_history, ensure_ascii=False, indent=2))
print("这只是消息组织实验，不是实际模型对话。")

原消息数： 1
下一轮消息： [
  {
    "role": "user",
    "content": "我正在学习大模型。"
  },
  {
    "role": "assistant",
    "content": "我们可以从分词器开始。"
  },
  {
    "role": "user",
    "content": "下一步学习什么？"
  }
]
这只是消息组织实验，不是实际模型对话。


## 14. 用脚本运行与排错

```bash
# 默认离线检查，不会联网或下载
python lesson_28_real_model_tokenizer_and_chat_template.py --checks-only

# 只下载分词器/配置
python lesson_28_real_model_tokenizer_and_chat_template.py --inspect

# 单条、短输出；GPU不可用直接报错
python lesson_28_real_model_tokenizer_and_chat_template.py --run --device cuda --dtype float16

# CPU可用但更慢，沿用FP32
python lesson_28_real_model_tokenizer_and_chat_template.py --run --device cpu

# 缓存完整后，禁止联网查找
python lesson_28_real_model_tokenizer_and_chat_template.py --run --local-files-only

# 左补齐批量对照，显存够用后再试
python lesson_28_real_model_tokenizer_and_chat_template.py --run --batch-demo --device cuda
```

也可用 `--sample` 开启本课明确的采样设置，`--prompt "你的问题"` 更换输入，或 `--messages-file history.json` 读取 UTF-8 JSON 消息列表。

| 现象 | 首先检查 |
|---|---|
| `KeyError: qwen3` | Transformers 太旧，或 Notebook 和终端不在同一环境 |
| 找不到模型/下载失败 | 网络、仓库名、固定 revision；本地模式是否已有完整文件 |
| CUDA不可用 | `torch.cuda.is_available()`，不要先怪模型 |
| OOM | 关闭其他权重副本，batch=1，缩短输入/输出，CUDA用16位；本课未量化 |
| 回答含提示原文 | 是否只切出了新增 token；短样本是否按 padded 宽度切片 |
| 能运行但回答异常 | 是否匹配 tokenizer/template、mode、EOS，是否因预算提前终止 |
| 依赖导入栈出现无关音频/视觉包错误 | 查看完整栈，优先用独立环境定位；不要改成 `trust_remote_code=True` |

本课不下载“替代的更小模型”冒充指定模型，也不在失败时自动编一段回答。没有自动执行任何生成的 Python 代码。

## 15. 独立练习：先写自己的版本，再打开开关

### A. 构造聊天批次

实现 `prepare_chat_ex(tokenizer, conversations)`：采用配套模板、生成前缀、非thinking模式、左补齐、不添加第二份特殊 token，并返回 `input_ids` 与 `attention_mask`。

可以调用已有 `validate_messages`，但不要直接把整个工作转交给 `prepare_chat_batch`。

In [23]:
RUN_EXERCISE_A = False

def prepare_chat_ex(tokenizer, conversations):
    raise NotImplementedError("自己实现模板与编码，不直接调用 prepare_chat_batch")

if RUN_EXERCISE_A:
    actual_inputs = prepare_chat_ex(FixtureTokenizer(), fixture_conversations)
    expected_inputs, _ = prepare_chat_batch(FixtureTokenizer(), fixture_conversations)
    assert torch.equal(actual_inputs['input_ids'], expected_inputs['input_ids'])
    assert torch.equal(actual_inputs['attention_mask'], expected_inputs['attention_mask'])
    print("练习A的教学替身检查通过；真实tokenizer需另开开关验证。")

### B. 只取新增编号

实现 `new_ids_ex(sequences, input_ids)`。校验前缀，沿所有样本共享的 padded 宽度截取。

已提供的反例：有效输入长度为2的那条记录，仍然需要跳过4列输入。

In [24]:
RUN_EXERCISE_B = False

def new_ids_ex(sequences, input_ids):
    raise NotImplementedError("不要使用 attention_mask.sum 作为切片起点")

if RUN_EXERCISE_B:
    x = torch.tensor([[0, 0, 10, 11], [20, 21, 22, 23]])
    y = torch.cat((x, torch.tensor([[31, 2], [32, 2]])), dim=1)
    assert new_ids_ex(y, x).tolist() == [[31, 2], [32, 2]]
    assert new_ids_ex(x, x).shape == (2, 0)
    print("练习B通过。")

### C. 准备下一轮对话

实现 `next_turn_ex(history, assistant_answer, user_question)`。保留旧历史，按正确 role 追加两条新消息，不原地改写原始 history。

进一步思考：只返回“用户最新问题”的版本为什么不再是同一个任务？

In [25]:
RUN_EXERCISE_C = False

def next_turn_ex(history, assistant_answer, user_question):
    raise NotImplementedError("保留角色、全部历史，且不修改原始对象")

if RUN_EXERCISE_C:
    original = [{"role": "user", "content": "记住这个词：向量"}]
    before = copy.deepcopy(original)
    result = next_turn_ex(original, "好的", "刚才的词是什么？")
    assert result == extend_conversation(before, "好的", "刚才的词是什么？")
    result[0]['content'] = "改动副本"
    assert original == before
    print("练习C通过。")

---
## 16. 参考答案（完成练习后再查看）

三个答案使用不同函数名，不会覆盖你的代码。它们依赖本讲义已经定义的基础校验与 PyTorch，不需要真实模型。

In [26]:
def prepare_chat_answer(tokenizer, conversations):
    if tokenizer.padding_side != "left":
        raise ValueError("生成使用左补齐")
    if not conversations:
        raise ValueError("至少需要一个对话")
    texts = []
    for history in conversations:
        clean = validate_messages(history)
        text = tokenizer.apply_chat_template(clean, tokenize=False,
                    add_generation_prompt=True, enable_thinking=False)
        texts.append(text)
    result = tokenizer(texts, add_special_tokens=False, padding=True,
                       truncation=False, return_tensors="pt", return_attention_mask=True)
    batch = {name: result[name] for name in ("input_ids", "attention_mask")}
    validate_left_padded_batch(batch)
    check_budget(batch['input_ids'].shape[1], 512, 64, 2048)
    return batch


def new_ids_answer(sequences, input_ids):
    if sequences.ndim != 2 or input_ids.ndim != 2:
        raise ValueError("必须是二维张量")
    width = input_ids.shape[1]
    if sequences.shape[0] != input_ids.shape[0] or sequences.shape[1] < width:
        raise ValueError("形状不匹配")
    if not torch.equal(sequences[:, :width], input_ids):
        raise ValueError("输入前缀不一致")
    return sequences[:, width:]


def next_turn_answer(history, assistant_answer, user_question):
    clean = validate_messages(history)
    if not isinstance(assistant_answer, str) or not isinstance(user_question, str):
        raise TypeError("新消息必须是字符串")
    clean.append({"role": "assistant", "content": assistant_answer})
    clean.append({"role": "user", "content": user_question})
    return validate_messages(clean)

In [27]:
# 参考答案的独立检查，不调用待补全的 exercise 函数。
a = prepare_chat_answer(FixtureTokenizer(), fixture_conversations)
b, _ = prepare_chat_batch(FixtureTokenizer(), fixture_conversations)
assert all(torch.equal(a[k], b[k]) for k in a)
x = torch.tensor([[0, 7], [7, 8]])
y = torch.cat((x, torch.tensor([[33, 2], [34, 2]])), dim=1)
assert new_ids_answer(y, x).tolist() == [[33, 2], [34, 2]]
assert new_ids_answer(x, x).shape == (2, 0)
old = [{"role": "user", "content": "x"}]
answer = next_turn_answer(old, "a", "b")
answer[0]['content'] = "changed"
assert old == [{"role": "user", "content": "x"}]
assert [m['role'] for m in answer] == ['user', 'assistant', 'user']
print("A/B/C 参考答案的离线检查通过。")

A/B/C 参考答案的离线检查通过。


## 17. 本课验收

学习完成时，你应当能解释：

- 为什么词表大小相同还不足以证明兼容，为什么不应该替换成第27课的 BPE；
- 模板后的文本、原始消息与实际 input_ids 的关系；
- 为什么右补齐的训练习惯不能直接照搬到 decoder-only 标准批量生成；
- `max_new_tokens`、EOS 和 `attention_mask` 的区别；
- 为什么 `head_dim` 应读配置，以及 Q 投影的输出宽度可以不同于 hidden_size；
- 真实模型下载成功、推理成功、答案正确，是三个不同的验收层次。

**真实任务验收**：在本机完成一条短提示，保留 `manifest.json`、`inspection.json`、`model.json`、`forward.json` 与 `generations.json`。解释实际设备/精度、输入token数、停止原因。不要把教学替身结果当作真实验收结果。

后续进入 **第29课：真实模型的固定题集与自动评测**。把“能生成”推进到“在同一输入、模板和预算下可比较”，覆盖输出格式、简单算术、文本抽取与代码题的验收边界，然后为量化和 LoRA 的效果比较建立基线。

## 18. 官方来源与本文件的验证范围

本课接口参考 Transformers v4.57.3 文档与当前模型卡，固定模型仓库 commit。来源用于核对具体 API 与模型事实，不代表代码已在所有版本/设备验证。

- **S1**：[Qwen3-0.6B 模型卡](https://huggingface.co/Qwen/Qwen3-0.6B)：模型范围、thinking开关、采样建议。
- **S2**：[固定 revision 的 config](https://huggingface.co/Qwen/Qwen3-0.6B/blob/c1899de289a04d12100db370d81485cdf75e47ca/config.json)：真实结构参数。
- **S3**：[Hugging Face Chat templates](https://huggingface.co/docs/transformers/chat_templating)：模板、生成前缀与重复特殊token。
- **S4**：[Text generation](https://huggingface.co/docs/transformers/v4.57.3/en/llm_tutorial)：decoder-only生成与padding。
- **S5**：[GenerationConfig](https://huggingface.co/docs/transformers/v4.57.3/en/main_classes/text_generation)：长度、EOS、生成设置。
- **S6**：[加载模型](https://huggingface.co/docs/transformers/v4.57.3/en/main_classes/model)：revision、dtype、local_files_only等。
- **S7**：[Qwen3实现](https://github.com/huggingface/transformers/blob/v4.57.3/src/transformers/models/qwen3/modeling_qwen3.py)：显式head_dim、Q/K norm、投影和输出头。
- **S8**：[tokenizer配置](https://huggingface.co/Qwen/Qwen3-0.6B/blob/c1899de289a04d12100db370d81485cdf75e47ca/tokenizer_config.json) 与 [generation配置](https://huggingface.co/Qwen/Qwen3-0.6B/blob/c1899de289a04d12100db370d81485cdf75e47ca/generation_config.json)。

**本次已执行的范围**：离线全部单元、74项逻辑/教学替身检查、参考答案；真实权重加载与 CUDA 未执行，条件单元明确输出“未执行”。不提供虚构的Qwen回答或性能结果。